# Scenario Builder

This notebook creates planning scenario bundles only. It reads site instances from `points.csv`, defines profiles and connectivity rules with `PlanningScenario`, then saves each scenario as TOML plus a companion `.nodes.csv`.

Run this notebook when you want to create or update reusable inputs for the execution notebook.

In [1]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from cisei_lib.planners import PlanningScenario

POINTS_PATH = NOTEBOOK_DIR / "points.csv"
OUT_DIR = NOTEBOOK_DIR / "scenario_exports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WORKING_CRS = "EPSG:31982"
CONNECTED_SITES = {"torre", "itallia"}
RELAY_SITES = {"saturno", "deneka"}

## Site Instances

The input CSV is intentionally simple. The notebook converts it to the standard instance table consumed by `PlanningScenario`: `site_id`, coordinates, optional `mount_height_m`, and later `device_profile`.

In [2]:
points = pd.read_csv(POINTS_PATH)
points = (
    points.rename(columns={"id": "site_id", "ant_h": "mount_height_m"})
    .assign(
        site_id=lambda df: df["site_id"].astype(str).str.strip(),
        lat=lambda df: df["lat"].astype(float),
        lon=lambda df: df["lon"].astype(float),
        mount_height_m=lambda df: df["mount_height_m"].astype(float),
    )
)
points

,site_id,lat,lon,mount_height_m
0,copel,-25.432731,-49.339207,7.0
1,tenis,-25.425357,-49.292122,7.0
2,praca_29,-25.427624,-49.285643,7.0
3,torre,-25.423532,-49.293992,100.0
4,woodland,-25.426584,-49.330740,7.0
5,deneka,-25.423934,-49.325575,7.0
6,cemiterio,-25.420709,-49.313294,7.0
7,saturno,-25.427867,-49.343001,7.0
8,decatlon,-25.428662,-49.322405,7.0
9,shopping,-25.435108,-49.317711,7.0


## Shared Profiles

These profiles are reused by the examples. A connected device is already attached to the backbone and receives `rank = 0` for RPL. Field devices are unconnected and only become relays when their device profile explicitly allows routing.

In [3]:
def add_common_lte_profiles(scenario: PlanningScenario) -> None:
    scenario.add_antenna_profile(
        "lte_tower_omni",
        {
            "kind": "omni",
            "description": "LTE tower omni antenna",
            "gain_dbi": 14.0,
        },
    )
    scenario.add_antenna_profile(
        "lte_leaf_omni",
        {
            "kind": "omni",
            "description": "LTE leaf omni antenna",
            "gain_dbi": 6.0,
        },
    )

    scenario.add_interface_profile(
        "lte_tower",
        {
            "tech": "lte",
            "freq_mhz": 400.0,
            "tx_power_dbm": 43.0,
            "antenna_id": "lte_tower_omni",
            "can_relay": False,
        },
    )
    scenario.add_interface_profile(
        "lte_leaf",
        {
            "tech": "lte",
            "freq_mhz": 400.0,
            "tx_power_dbm": 20.0,
            "antenna_id": "lte_leaf_omni",
            "can_relay": False,
        },
    )

    scenario.add_device_profile(
        "connected_lte_tower",
        {
            "kind": "tower",
            "connected": True,
            "can_route": True,
            "rank": 0.0,
            "interfaces": ["lte_tower"],
        },
    )
    scenario.add_device_profile(
        "lte_leaf_device",
        {
            "kind": "field",
            "connected": False,
            "can_route": False,
            "interfaces": ["lte_leaf"],
        },
    )

    scenario.add_metric_spec("lte", {"spec": "lte_tower_node"})

## Scenario 1: Single-Interface LTE

All sites have one LTE interface. `torre` and `itallia` are connected sites; every other site is a non-routing LTE leaf.

In [4]:
single = PlanningScenario(working_crs=WORKING_CRS)
add_common_lte_profiles(single)
single.add_connectivity_rule(
    {
        "kind": "rf",
        "tech": "lte",
        "source": "connected",
        "destination": "unconnected",
    }
)

single_instances = points.copy()
single_instances["device_profile"] = single_instances["site_id"].apply(
    lambda site_id: "connected_lte_tower" if site_id in CONNECTED_SITES else "lte_leaf_device"
)

single.add_node_instances(single_instances)
print("validation errors:", single.validate())
single.instance_table()

validation errors: []


,site_id,device_profile,kind,lat,lon,x,y,mount_height_m,connected,can_route,rank
0,copel,lte_leaf_device,field,-25.432731,-49.339207,667013.250467,7.186095e+06,7.0,False,False,inf
1,tenis,lte_leaf_device,field,-25.425357,-49.292122,671759.559806,7.186852e+06,7.0,False,False,inf
2,praca_29,lte_leaf_device,field,-25.427624,-49.285643,672408.065400,7.186593e+06,7.0,False,False,inf
3,torre,connected_lte_tower,tower,-25.423532,-49.293992,671574.031187,7.187057e+06,100.0,True,True,0.0
4,woodland,lte_leaf_device,field,-25.426584,-49.330740,667873.384644,7.186766e+06,7.0,False,False,inf
5,deneka,lte_leaf_device,field,-25.423934,-49.325575,668396.573489,7.187053e+06,7.0,False,False,inf
6,cemiterio,lte_leaf_device,field,-25.420709,-49.313294,669636.488343,7.187394e+06,7.0,False,False,inf
7,saturno,lte_leaf_device,field,-25.427867,-49.343001,666638.395068,7.186639e+06,7.0,False,False,inf
8,decatlon,lte_leaf_device,field,-25.428662,-49.322405,668708.855680,7.186525e+06,7.0,False,False,inf
9,shopping,lte_leaf_device,field,-25.435108,-49.317711,669172.039893,7.185805e+06,7.0,False,False,inf


In [5]:
single_toml, single_csv = single.save_bundle(
    OUT_DIR / "graph_planner_single_interface.toml"
)

print("TOML:", single_toml)
print("instances CSV:", single_csv)

TOML: /workspaces/network_service/tests/jupyter/scenario_exports/graph_planner_single_interface.toml
instances CSV: /workspaces/network_service/tests/jupyter/scenario_exports/graph_planner_single_interface.nodes.csv


## Scenario 2: LTE + Node-To-Node Relays

Most sites remain pure LTE leaves. Only `saturno` and `deneka` receive an n2n interface and can route internally between LTE and n2n. This keeps the graph small enough to inspect while testing multi-interface planning.

In [6]:
two_if = PlanningScenario(working_crs=WORKING_CRS)
add_common_lte_profiles(two_if)

two_if.add_antenna_profile(
    "n2n_omni",
    {
        "kind": "omni",
        "description": "Node-to-node omni antenna",
        "gain_dbi": 3.0,
    },
)
two_if.add_interface_profile(
    "n2n_relay",
    {
        "tech": "n2n",
        "freq_mhz": 915.0,
        "tx_power_dbm": 23.0,
        "antenna_id": "n2n_omni",
        "can_relay": True,
    },
)
two_if.add_device_profile(
    "dual_lte_n2n_relay",
    {
        "kind": "field",
        "connected": False,
        "can_route": True,
        "interfaces": ["lte_leaf", "n2n_relay"],
    },
)

two_if.add_metric_spec("n2n", {"spec": "default"})
two_if.add_connectivity_rule(
    {
        "kind": "rf",
        "tech": "lte",
        "source": "connected",
        "destination": "unconnected",
    }
)
two_if.add_connectivity_rule(
    {
        "kind": "rf",
        "tech": "n2n",
        "source": "relay",
        "destination": "any",
        "degree": 3,
        "limit": 1000.0,
    }
)
two_if.add_connectivity_rule(
    {
        "kind": "internal",
        "enabled_when_device_can_route": True,
        "metric": 0.0,
    }
)

'connectivity_2'

In [7]:
two_if_instances = points.copy()

def two_if_profile(site_id: str) -> str:
    if site_id in CONNECTED_SITES:
        return "connected_lte_tower"
    if site_id in RELAY_SITES:
        return "dual_lte_n2n_relay"
    return "lte_leaf_device"

two_if_instances["device_profile"] = two_if_instances["site_id"].apply(two_if_profile)

two_if.add_node_instances(two_if_instances)
print("validation errors:", two_if.validate())
two_if.instance_table()

validation errors: []


,site_id,device_profile,kind,lat,lon,x,y,mount_height_m,connected,can_route,rank
0,copel,lte_leaf_device,field,-25.432731,-49.339207,667013.250467,7.186095e+06,7.0,False,False,inf
1,tenis,lte_leaf_device,field,-25.425357,-49.292122,671759.559806,7.186852e+06,7.0,False,False,inf
2,praca_29,lte_leaf_device,field,-25.427624,-49.285643,672408.065400,7.186593e+06,7.0,False,False,inf
3,torre,connected_lte_tower,tower,-25.423532,-49.293992,671574.031187,7.187057e+06,100.0,True,True,0.0
4,woodland,lte_leaf_device,field,-25.426584,-49.330740,667873.384644,7.186766e+06,7.0,False,False,inf
5,deneka,dual_lte_n2n_relay,field,-25.423934,-49.325575,668396.573489,7.187053e+06,7.0,False,True,inf
6,cemiterio,lte_leaf_device,field,-25.420709,-49.313294,669636.488343,7.187394e+06,7.0,False,False,inf
7,saturno,dual_lte_n2n_relay,field,-25.427867,-49.343001,666638.395068,7.186639e+06,7.0,False,True,inf
8,decatlon,lte_leaf_device,field,-25.428662,-49.322405,668708.855680,7.186525e+06,7.0,False,False,inf
9,shopping,lte_leaf_device,field,-25.435108,-49.317711,669172.039893,7.185805e+06,7.0,False,False,inf


In [8]:
two_if_toml, two_if_csv = two_if.save_bundle(
    OUT_DIR / "graph_planner_two_interface.toml"
)

print("TOML:", two_if_toml)
print("instances CSV:", two_if_csv)

TOML: /workspaces/network_service/tests/jupyter/scenario_exports/graph_planner_two_interface.toml
instances CSV: /workspaces/network_service/tests/jupyter/scenario_exports/graph_planner_two_interface.nodes.csv


## Scenario 3: Two-Point Antenna Function Test

This scenario keeps only `torre` and `guaira`. Both interfaces use `model_id`, so metric computation exercises the library-backed antenna gain function instead of fixed `gain_dbi`.


In [ ]:
antenna_test = PlanningScenario(working_crs=WORKING_CRS)

antenna_test.add_antenna_profile(
    "tower_pctel_boa9028",
    {
        "kind": "omni",
        "model_id": "pctel_boa9028",
        "description": "Library-backed PCTEL BOA9028 at the connected site",
        "azimuth_deg": 0.0,
        "downtilt_deg": 0.0,
    },
)
antenna_test.add_antenna_profile(
    "leaf_pctel_boa9028",
    {
        "kind": "omni",
        "model_id": "pctel_boa9028",
        "description": "Library-backed PCTEL BOA9028 at the field site",
        "azimuth_deg": 0.0,
        "downtilt_deg": 0.0,
    },
)
antenna_test.add_interface_profile(
    "tower_915_library",
    {
        "tech": "antenna_test",
        "freq_mhz": 915.0,
        "tx_power_dbm": 30.0,
        "antenna_id": "tower_pctel_boa9028",
        "can_relay": False,
    },
)
antenna_test.add_interface_profile(
    "leaf_915_library",
    {
        "tech": "antenna_test",
        "freq_mhz": 915.0,
        "tx_power_dbm": 20.0,
        "antenna_id": "leaf_pctel_boa9028",
        "can_relay": False,
    },
)
antenna_test.add_device_profile(
    "connected_antenna_test_site",
    {
        "kind": "tower",
        "connected": True,
        "can_route": True,
        "rank": 0.0,
        "interfaces": ["tower_915_library"],
    },
)
antenna_test.add_device_profile(
    "field_antenna_test_site",
    {
        "kind": "field",
        "connected": False,
        "can_route": False,
        "interfaces": ["leaf_915_library"],
    },
)
antenna_test.add_metric_spec("antenna_test", {"spec": "default"})
antenna_test.add_connectivity_rule(
    {
        "kind": "rf",
        "tech": "antenna_test",
        "source": "connected",
        "destination": "unconnected",
    }
)

antenna_instances = points[points["site_id"].isin(["torre", "guaira"])].copy()
antenna_instances["device_profile"] = antenna_instances["site_id"].map(
    {
        "torre": "connected_antenna_test_site",
        "guaira": "field_antenna_test_site",
    }
)

antenna_test.add_node_instances(antenna_instances)
print("validation errors:", antenna_test.validate())
antenna_test.instance_table()


In [ ]:
antenna_test_toml, antenna_test_csv = antenna_test.save_bundle(
    OUT_DIR / "graph_planner_antenna_function_test.toml"
)

print("TOML:", antenna_test_toml)
print("instances CSV:", antenna_test_csv)


## Scenario 4: CellPlanner Sector Antenna Scenario

This scenario is complete: profiles and node instances are both stored in the bundle. `CellPlanner` can load it directly without assigning default profiles. Sector antennas use `model_id` so metric computation uses the antenna library gain function.


In [ ]:
cell_sector = PlanningScenario(working_crs=WORKING_CRS)

for antenna_id, azimuth in {
    "lte_sector_0": 0.0,
    "lte_sector_120": 120.0,
    "lte_sector_240": 240.0,
}.items():
    cell_sector.add_antenna_profile(
        antenna_id,
        {
            "kind": "sector",
            "model_id": "commscope_rv_65s_fvb",
            "description": f"CommScope RV-65S-FVB sector at {azimuth:.0f} deg",
            "azimuth_deg": azimuth,
            "downtilt_deg": 0.0,
            "beamwidth_deg": 120.0,
        },
    )

cell_sector.add_antenna_profile(
    "lte_client_omni",
    {
        "kind": "omni",
        "model_id": "pctel_boa9028",
        "description": "Library-backed client omni antenna",
        "azimuth_deg": 0.0,
        "downtilt_deg": 0.0,
    },
)

for interface_id, antenna_id in {
    "lte_sector_0": "lte_sector_0",
    "lte_sector_120": "lte_sector_120",
    "lte_sector_240": "lte_sector_240",
}.items():
    cell_sector.add_interface_profile(
        interface_id,
        {
            "tech": "lte",
            "freq_mhz": 915.0,
            "tx_power_dbm": 43.0,
            "antenna_id": antenna_id,
            "can_relay": False,
        },
    )

cell_sector.add_interface_profile(
    "lte_client",
    {
        "tech": "lte",
        "freq_mhz": 915.0,
        "tx_power_dbm": 20.0,
        "antenna_id": "lte_client_omni",
        "can_relay": False,
    },
)

cell_sector.add_device_profile(
    "cell_sector_3x120",
    {
        "kind": "tower",
        "connected": True,
        "can_route": True,
        "rank": 0.0,
        "interfaces": ["lte_sector_0", "lte_sector_120", "lte_sector_240"],
    },
)
cell_sector.add_device_profile(
    "cell_sector_2x120",
    {
        "kind": "tower",
        "connected": True,
        "can_route": True,
        "rank": 0.0,
        "interfaces": ["lte_sector_120", "lte_sector_240"],
    },
)
cell_sector.add_device_profile(
    "lte_leaf",
    {
        "kind": "field",
        "connected": False,
        "can_route": False,
        "interfaces": ["lte_client"],
    },
)

cell_sector.add_metric_spec("lte", {"spec": "lte_tower_node"})


In [ ]:
cell_instances = points.copy()

def cell_profile(site_id: str) -> str:
    if site_id == "torre":
        return "cell_sector_3x120"
    if site_id == "itallia":
        return "cell_sector_2x120"
    return "lte_leaf"

cell_instances["device_profile"] = cell_instances["site_id"].apply(cell_profile)

cell_sector.add_node_instances(cell_instances)
print("validation errors:", cell_sector.validate())
cell_sector.instance_table()


In [ ]:
cell_sector_toml, cell_sector_csv = cell_sector.save_bundle(
    OUT_DIR / "cell_planner_sector_antenna.toml"
)

print("TOML:", cell_sector_toml)
print("instances CSV:", cell_sector_csv)


## Reload Check

This checks that the saved TOML files can load their companion `.nodes.csv` files automatically.

In [9]:
loaded_single = PlanningScenario.from_toml(single_toml)
loaded_two_if = PlanningScenario.from_toml(two_if_toml)
loaded_antenna_test = PlanningScenario.from_toml(antenna_test_toml)
loaded_cell_sector = PlanningScenario.from_toml(cell_sector_toml)

print("single nodes:", len(loaded_single.site_nodes), "errors:", loaded_single.validate())
print("two-interface nodes:", len(loaded_two_if.site_nodes), "errors:", loaded_two_if.validate())
print("antenna-test nodes:", len(loaded_antenna_test.site_nodes), "errors:", loaded_antenna_test.validate())
print("cell-sector nodes:", len(loaded_cell_sector.site_nodes), "errors:", loaded_cell_sector.validate())


single nodes: 13 errors: []
two-interface nodes: 13 errors: []
